# Faruq-v3 — all top controls sequential Kaggle

Satu notebook menjalankan delapan arm secara berurutan: FCT0, AF2R0, AF2R1, dan AF2CAL3 untuk seed 123/2026. Dataset test tidak disertakan. Output dapat dipulihkan dari Saved Version sebelumnya bila output itu dipasang sebagai Kaggle Input.

Kaggle Inputs wajib memuat `faruq-development-v3-grouped.tar.bin` dan bundle kecil dengan `top_controls_kaggle_manifest.json`. Aktifkan GPU dan Internet sebelum menjalankan sebagai Saved Version.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
manifests=sorted(INPUT.rglob('top_controls_kaggle_manifest.json'))
archives=sorted(INPUT.rglob('faruq-development-v3-grouped.tar.bin'))
if len(manifests)!=1 or len(archives)!=1:
    raise FileNotFoundError(f'STOP CEPAT: butuh tepat satu manifest dan archive; manifest={manifests}, archive={archives}')
print('INPUT PREFLIGHT PASS'); print('MANIFEST:',manifests[0]); print('ARCHIVE:',archives[0])


In [ ]:
import importlib, os, shutil, subprocess, sys
REPO=WORK/'coffee-bean-detection'; BRANCH='codex/top-controls-multiseed-confirmation'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
os.chdir(REPO)
SRC=REPO/'src'
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name,None)
importlib.invalidate_caches()
from coffee_detector.experiments.prepare_top_controls_kaggle import (
    ensure_run_contract, prepare_top_controls_kaggle_input,
    restore_top_control_kaggle_run, run_directory)
DATA,ARTIFACTS,INPUT_CONTRACT=prepare_top_controls_kaggle_input(INPUT,WORK)
assert INPUT_CONTRACT['decision']=='PASS' and INPUT_CONTRACT['test_images_accessed'] is False
assert not (DATA/'test').exists(), 'STOP: test tidak boleh tersedia'
print('PACKAGE/DATA CONTRACT PASS:',DATA)


In [ ]:
ARMS=('FCT0','AF2R0','AF2R1','AF2CAL3'); SEEDS=(123,2026)
OUTPUT=WORK/'faruq-v3-top-controls-paired-confirmation-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
GROUPED=DATA/'faruq_grouped_summary.json'
def source_for(arm,seed):
    prefix='STB1' if arm=='FCT0' else 'AF2'
    return ARTIFACTS[f'{prefix}_seed{seed}_best.pt']
restored={}
for arm in ARMS:
    for seed in SEEDS:
        restored[f'{arm}_{seed}']=restore_top_control_kaggle_run(INPUT,OUTPUT,arm=arm,seed=seed,source_checkpoint=source_for(arm,seed))
print('RESTORED:',{key:str(value) if value else None for key,value in restored.items()})


In [ ]:
import csv, json, time
from coffee_detector.af2r.audit import run_af2r_static_audit
from coffee_detector.af2cal.audit import run_af2cal_static_audit
LOGS=OUTPUT/'logs'; LOGS.mkdir(parents=True,exist_ok=True)
RESULTS={}
def run_one(arm,seed):
    source=source_for(arm,seed)
    ensure_run_contract(OUTPUT,arm=arm,seed=seed,source_checkpoint=source)
    static=OUTPUT/'static_audits'/f'{arm}_seed{seed}.json'; static.parent.mkdir(parents=True,exist_ok=True)
    if arm in {'AF2R0','AF2R1'}:
        audit=run_af2r_static_audit(source,static,device='cuda:0')
        if audit['decision']!='PASS': raise RuntimeError(f'Static AF2R gagal: {arm}/{seed}')
        module='coffee_detector.experiments.run_faruq_v3_af2r_arm'
        extra=['--arm',arm,'--af2-checkpoint',str(source),'--static-audit',str(static)]
    elif arm=='AF2CAL3':
        audit=run_af2cal_static_audit(source,static,device='cuda:0')
        if audit['decision']!='PASS': raise RuntimeError(f'Static AF2CAL gagal: {arm}/{seed}')
        module='coffee_detector.experiments.run_faruq_v3_af2cal_arm'
        extra=['--arm',arm,'--af2-checkpoint',str(source),'--static-audit',str(static)]
    else:
        module='coffee_detector.experiments.run_faruq_v3_fct0_confirmation_arm'
        extra=['--stb-confirmation',str(ARTIFACTS['stb_capacity_paired_confirmation.json']),'--stb-checkpoint',str(source)]
    command=[sys.executable,'-u','-m',module,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--output-root',str(OUTPUT),'--seed',str(seed),'--device','0','--authorize-training',*extra]
    log=LOGS/f'{arm}_seed{seed}_run.log'; run_dir=run_directory(OUTPUT,arm,seed)
    print(f'\nSTART/RESUME {arm} seed {seed} | {log}',flush=True)
    with log.open('a',encoding='utf-8') as stream:
        process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
        shown=-1
        while process.poll() is None:
            epochs=0; csv_path=run_dir/'results.csv'
            if csv_path.is_file():
                try:
                    with csv_path.open(newline='',encoding='utf-8') as handle: epochs=len(list(csv.DictReader(handle)))
                except Exception: pass
            if epochs!=shown: print(f'{arm} seed {seed}: {epochs} epoch tercatat',flush=True); shown=epochs
            time.sleep(30)
    if process.returncode:
        print('\n'.join(log.read_text(errors='replace').splitlines()[-150:]))
        raise RuntimeError(f'{arm} seed {seed} gagal: {process.returncode}')
    result_path=OUTPUT/'val_reports'/f'{arm}_seed{seed}_result.json'
    if not result_path.is_file(): raise FileNotFoundError(result_path)
    result=json.loads(result_path.read_text()); metrics=result['metrics']
    headline={key:metrics[key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}
    print('SELESAI',arm,seed,headline,flush=True); return result_path
for arm in ARMS:
    for seed in SEEDS:
        RESULTS[f'{arm}_{seed}']=run_one(arm,seed)


In [ ]:
from coffee_detector.experiments.run_faruq_v3_top_controls_confirmation_decision import run_faruq_v3_top_controls_confirmation_decision
def pair(arm): return tuple(RESULTS[f'{arm}_{seed}'] for seed in SEEDS)
decision=run_faruq_v3_top_controls_confirmation_decision(
    ARTIFACTS['stb_capacity_paired_confirmation.json'],
    ARTIFACTS['af2_igem_paired_confirmation.json'],
    ARTIFACTS['af2_continuation_paired_confirmation.json'],
    ARTIFACTS['FCT0_seed42_val.json'],
    REPO/'docs/evidence/FARUQ_V3_AF2R_SCREENING_2026-08-17.json',
    REPO/'docs/evidence/FARUQ_V3_AF2_CHANNEL_CALIBRATION_SCREENING_2026-08-17.json',
    pair('FCT0'),pair('AF2R0'),pair('AF2R1'),pair('AF2CAL3'),
    OUTPUT/'val_reports/top_controls_paired_confirmation.json')
print('RETAINED:',decision['retained'])
for arm,row in decision['comparisons'].items():
    print(arm,row['decision'],row['criteria'])
print('TEST:',decision['test_opened'])


In [ ]:
archive=Path(shutil.make_archive(str(WORK/'faruq-v3-top-controls-paired-confirmation-v1-output'),'zip',root_dir=OUTPUT))
manifest={
    'format':'coffee_detector.top_controls.kaggle_output.v1',
    'branch':BRANCH,'commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(),
    'input_contract':INPUT_CONTRACT,'retained':decision['retained'],
    'summary':str(OUTPUT/'val_reports/top_controls_paired_confirmation.json'),
    'zip':str(archive),'test_images_accessed':False}
(WORK/'top_controls_kaggle_output_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('OUTPUT ZIP:',archive,archive.stat().st_size,'bytes')
print('Simpan sebagai Kaggle Version. Jika runtime terputus, attach output version ini sebagai Input lalu jalankan ulang.')
